# Chapter 12 Computational Lab
## Conditional Expectation: Information, Prediction and Variance Decomposition

This notebook accompanies Chapter 12 of *Probability Theory with Python and AI*.

The chapter treats conditional expectation as an **information-dependent random variable**, not merely as an average inside a familiar conditional distribution.

The conceptual order is important:

$$
\text{events and partitions}
\longrightarrow
\text{abstract conditional expectation}
\longrightarrow
\text{properties and convergence}
\longrightarrow
\text{Jensen, variance and prediction}
\longrightarrow
\text{conditioning on }Y
\longrightarrow
\text{conditional densities}.
$$

### Learning goals

By the end of the lab you should be able to:

1. explain why a conditional expectation given information is itself a random variable;
2. compute conditional means on events and finite partitions;
3. state the general definition using $\mathbb E[X\mathbf 1_A]$;
4. explain the equivalent integral formulation and the role of Radon--Nikodym;
5. work with conditional probability as an indicator conditional expectation;
6. use linearity, monotonicity, mean preservation, idempotence and contraction;
7. recognize the conditional forms of MCT, Fatou and DCT;
8. use the testing identity and “take out what is known”;
9. apply the tower property with the required nesting of information;
10. interpret the law of total expectation as “split into cases and recombine”;
11. work with filtrations, adapted processes and time-indexed conditional predictions;
12. understand conditioning on independent information;
13. apply conditional Jensen and $L^p$ contraction;
14. compute conditional variance and use the law of total variance;
15. use conditional covariance and the law of total covariance;
16. interpret conditional expectation as the MMSE predictor;
17. understand conditioning on $Y$ as conditioning on $\sigma(Y)$;
18. compute countable and density-based conditional means;
19. derive the bivariate-normal conditional law;
20. explain Galton's regression toward the mean;
21. audit AI-generated claims about conditioning.

> **Notation policy.** Expectation notation is primary in the abstract theory. Integral notation is used when it exposes measure structure, especially in the Radon--Nikodym existence argument.


## 0. Setup

Finite examples are represented explicitly by outcomes, probabilities and information cells.


In [ ]:
from fractions import Fraction
import math
import random

import matplotlib.pyplot as plt
import numpy as np
import ipywidgets as widgets
from IPython.display import HTML, Math, Markdown, clear_output, display

try:
    from google.colab import output as colab_output
    colab_output.enable_custom_widget_manager()
except ImportError:
    pass


def fmt_fraction(x):
    x = Fraction(x)
    if x.denominator == 1:
        return str(x.numerator)
    return rf"\frac{{{x.numerator}}}{{{x.denominator}}}"


def normalize_probs(probs):
    probs = [Fraction(p) for p in probs]
    total = sum(probs, Fraction(0, 1))
    if total <= 0:
        raise ValueError("Total probability must be positive.")
    return [p / total for p in probs]


def expectation(values, probs):
    return sum(
        Fraction(v) * Fraction(p)
        for v, p in zip(values, probs)
    )


def conditional_mean_on_event(values, probs, mask):
    numerator = sum(
        Fraction(v) * Fraction(p)
        for v, p, keep in zip(values, probs, mask)
        if keep
    )
    denominator = sum(
        Fraction(p)
        for p, keep in zip(probs, mask)
        if keep
    )
    if denominator == 0:
        raise ZeroDivisionError("Elementary event conditioning requires positive probability.")
    return numerator / denominator


def conditional_expectation_partition(values, probs, labels):
    values = list(values)
    probs = [Fraction(p) for p in probs]
    labels = list(labels)

    unique = []
    for label in labels:
        if label not in unique:
            unique.append(label)

    cell_means = {}
    for label in unique:
        mask = [g == label for g in labels]
        mass = sum(p for p, keep in zip(probs, mask) if keep)
        if mass == 0:
            cell_means[label] = Fraction(0, 1)
        else:
            cell_means[label] = conditional_mean_on_event(values, probs, mask)

    ce = [cell_means[label] for label in labels]
    return ce, cell_means


def weighted_mse(target, predictor, probs):
    return sum(
        Fraction(p) * (Fraction(x)-Fraction(z))**2
        for x, z, p in zip(target, predictor, probs)
    )


def normal_pdf(x, mu=0.0, sigma=1.0):
    x = np.asarray(x, dtype=float)
    return np.exp(-0.5*((x-mu)/sigma)**2)/(sigma*math.sqrt(2*math.pi))


def bivariate_normal_conditional_mean(y, mu_x, mu_y, sigma_x, sigma_y, rho):
    return mu_x + rho*(sigma_x/sigma_y)*(y-mu_y)


def bivariate_normal_conditional_variance(sigma_x, rho):
    return sigma_x**2 * (1-rho**2)


def show_result(title, *latex_lines, note=None):
    display(HTML(
        f"<div style='border-left:5px solid;padding:8px 12px;margin:8px 0'>"
        f"<b>{title}</b></div>"
    ))
    for line in latex_lines:
        display(Math(line))
    if note:
        display(Markdown(note))


display(HTML(
    "<div style='padding:10px;border:1px solid'>"
    "<b>Setup complete.</b> Conditional-expectation tools are ready."
    "</div>"
))


## 1. Why a conditional mean must depend on information

Before new information is observed, $\mathbb E[X]$ is one number.

After information represented by a sub-$\sigma$-algebra $\mathcal G$ is revealed, the updated mean should depend only on what $\mathcal G$ can distinguish.

Therefore the general object

$$
\mathbb E[X\mid\mathcal G]
$$

must be a $\mathcal G$-measurable random variable.


### Fair die observed only as even or odd

Let $X$ be a fair die and let

$$
A=\{X\text{ is even}\}.
$$

The information is

$$
\mathcal G=\sigma(A)
=
\{\varnothing,A,A^c,\Omega\}.
$$

On $A$,

$$
\mathbb E[X\mid A]=4.
$$

On $A^c$,

$$
\mathbb E[X\mid A^c]=3.
$$

Hence

$$
\boxed{
\mathbb E[X\mid\mathcal G]
=
4\mathbf 1_A
+
3\mathbf 1_{A^c}.
}
$$


In [ ]:
die_values = [1,2,3,4,5,6]
die_probs = [Fraction(1,6)] * 6
parity = ["odd" if x % 2 else "even" for x in die_values]

ce_die, die_cell_means = conditional_expectation_partition(
    die_values,
    die_probs,
    parity,
)

display(Markdown(f"Cell means: **{die_cell_means}**"))
display(Markdown(f"Conditional-expectation values by die outcome: **{ce_die}**"))


In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.2))
ax.bar(["odd information cell", "even information cell"], [3,4])
ax.set_ylabel("conditional mean")
ax.set_ylim(0,5)
ax.set_title("Information reveals the cell, not the exact die outcome")
plt.show()


## 2. Conditioning on an event

If $X\in L^1$ and $P(A)>0$, define

$$
\boxed{
\mathbb E[X\mid A]
=
\frac{\mathbb E[X\mathbf 1_A]}{P(A)}.
}
$$

The condition $P(A)>0$ is essential for this elementary quotient.


In [ ]:
event_choices = widgets.SelectMultiple(
    options=[1,2,3,4,5,6],
    value=(2,4,6),
    description="A",
)
event_output = widgets.Output()


def update_event_conditioning(*_):
    with event_output:
        clear_output(wait=True)

        selected = set(event_choices.value)
        mask = [x in selected for x in die_values]

        if not any(mask):
            display(Markdown("**The selected event has probability zero in this finite model.**"))
            return

        mean = conditional_mean_on_event(die_values, die_probs, mask)
        mass = sum(p for p, keep in zip(die_probs, mask) if keep)

        display(Math(r"P(A)=" + fmt_fraction(mass)))
        display(Math(r"\mathbb E[X\mid A]=" + fmt_fraction(mean)))


event_choices.observe(update_event_conditioning, names="value")
display(widgets.VBox([event_choices, event_output]))
update_event_conditioning()


### From an event to the $\sigma$-algebra it generates

If

$$
0<P(A)<1,
$$

then

$$
\boxed{
\mathbb E[X\mid\sigma(A)]
=
\mathbb E[X\mid A]\mathbf 1_A
+
\mathbb E[X\mid A^c]\mathbf 1_{A^c}.
}
$$

The event-conditioned quantities are numbers; the $\sigma(A)$-conditioned quantity is a random variable.


## 3. Conditioning on a finite or countable partition

Let $\{A_i\}$ be a finite or countable measurable partition and let

$$
\mathcal G=\sigma(A_i:i\in I).
$$

On each positive-probability cell,

$$
\boxed{
\mathbb E[X\mid\mathcal G]
=
\frac{\mathbb E[X\mathbf 1_{A_i}]}{P(A_i)}
\quad\text{on }A_i.
}
$$

Thus conditional expectation replaces $X$ by its average on each observable information cell.


In [ ]:
partition_choice = widgets.Dropdown(
    options=[
        ("parity", "parity"),
        ("low/high", "lowhigh"),
        ("three pairs", "pairs"),
    ],
    value="parity",
    description="partition",
)
partition_output = widgets.Output()


def update_partition(*_):
    with partition_output:
        clear_output(wait=True)

        if partition_choice.value == "parity":
            labels = ["odd" if x%2 else "even" for x in die_values]
        elif partition_choice.value == "lowhigh":
            labels = ["low" if x <= 3 else "high" for x in die_values]
        else:
            labels = [f"pair{(x-1)//2+1}" for x in die_values]

        ce, means = conditional_expectation_partition(
            die_values,
            die_probs,
            labels,
        )

        display(Markdown(f"**Cell means:** {means}"))
        display(Markdown(f"**Conditional expectation on outcomes 1,...,6:** {ce}"))

        original_mean = expectation(die_values, die_probs)
        conditional_mean = expectation(ce, die_probs)

        display(Math(r"\mathbb E[X]=" + fmt_fraction(original_mean)))
        display(Math(
            r"\mathbb E[\mathbb E[X\mid\mathcal G]]="
            + fmt_fraction(conditional_mean)
        ))


partition_choice.observe(update_partition, names="value")
display(widgets.VBox([partition_choice, partition_output]))
update_partition()


## 4. General definition

Let $\mathcal G\subseteq\mathcal F$.

For a non-negative random variable or an integrable real-valued random variable $X$, a $\mathcal G$-measurable random variable $Y$ is a conditional expectation of $X$ given $\mathcal G$ if

$$
\boxed{
\mathbb E[Y\mathbf 1_A]
=
\mathbb E[X\mathbf 1_A]
\qquad
\text{for every }A\in\mathcal G.
}
$$

We write

$$
Y=\mathbb E[X\mid\mathcal G].
$$

For $X\ge0$, the conditional expectation may take the value $+\infty$.

For $X\in L^1$, it is required to be integrable.


### Equivalent integral language

For every non-negative or integrable $Z$,

$$
\mathbb E[Z\mathbf 1_A]
=
\int_A Z\,dP.
$$

Hence the defining identity is equivalently

$$
\int_A
\mathbb E[X\mid\mathcal G]\,dP
=
\int_A X\,dP,
\qquad
A\in\mathcal G.
$$

The notebook uses expectation notation as the main language, matching the chapter's current notation policy.


### Two distinct requirements

The definition contains two conceptually different conditions.

**Measurability:**

$$
\mathbb E[X\mid\mathcal G]
$$

may use no information outside $\mathcal G$.

**Testing identity:**

on every event that $\mathcal G$ can distinguish, the conditional expectation preserves the same expected mass as $X$.


## 5. Existence, uniqueness and versions

For every sub-$\sigma$-algebra $\mathcal G$:

- if $X\in L^1$, $\mathbb E[X\mid\mathcal G]\in L^1$ exists;
- if $X\ge0$, a non-negative extended-valued version exists;
- versions are unique only up to almost-sure equality.

The integrable existence proof applies Radon--Nikodym to the measures

$$
\nu_+(A)=\mathbb E[X^+\mathbf 1_A],
\qquad
\nu_-(A)=\mathbb E[X^-\mathbf 1_A].
$$

The non-negative case follows by truncating $X$ and using monotone convergence.


### Versions may differ on null sets

Conditional expectation is generally not determined pointwise.

If $Y$ and $Z$ are two versions of

$$
\mathbb E[X\mid\mathcal G],
$$

then

$$
P(Y=Z)=1.
$$

Statements involving conditional expectation are therefore almost-sure statements unless a specific version has been selected.


## 6. Conditional probability given information

For an event $B$,

$$
\boxed{
P(B\mid\mathcal G)
=
\mathbb E[\mathbf 1_B\mid\mathcal G].
}
$$

Thus conditional probability is a special case of conditional expectation.

It is itself a $\mathcal G$-measurable random variable in $[0,1]$.


### Die example

Let

$$
B=\{X=6\}
$$

and

$$
\mathcal G=\sigma(\{X\text{ is even}\}).
$$

Then

$$
\boxed{
P(B\mid\mathcal G)
=
\frac13
\mathbf 1_{\{X\text{ is even}\}}.
}
$$

If the information says “odd”, the conditional probability is $0$; if it says “even”, it is $1/3$.


In [ ]:
cond_prob_values = [
    Fraction(0,1) if x%2 else Fraction(1,3)
    for x in die_values
]

display(Markdown(
    f"Conditional probability values on die outcomes 1,...,6: **{cond_prob_values}**"
))
display(Math(
    r"\mathbb E[P(B\mid\mathcal G)]="
    + fmt_fraction(expectation(cond_prob_values, die_probs))
))
display(Math(r"P(B)=\frac16"))


## 7. Fundamental properties

For appropriate non-negative or integrable variables:

$$
\mathbb E[aX+bY\mid\mathcal G]
=
a\mathbb E[X\mid\mathcal G]
+
b\mathbb E[Y\mid\mathcal G],
$$

$$
X\le Y
\Longrightarrow
\mathbb E[X\mid\mathcal G]
\le
\mathbb E[Y\mid\mathcal G],
$$

$$
\boxed{
\mathbb E[\mathbb E[X\mid\mathcal G]]
=
\mathbb E[X],
}
$$

and if $X$ is already $\mathcal G$-measurable,

$$
\boxed{
\mathbb E[X\mid\mathcal G]=X.
}
$$


### Absolute-value bound and contraction

If $X\in L^1$,

$$
\boxed{
|\mathbb E[X\mid\mathcal G]|
\le
\mathbb E[|X|\mid\mathcal G].
}
$$

Therefore

$$
\boxed{
\|\mathbb E[X\mid\mathcal G]\|_1
\le
\|X\|_1.
}
$$

Conditional expectation is also idempotent:

$$
\boxed{
\mathbb E[
\mathbb E[X\mid\mathcal G]
\mid\mathcal G
]
=
\mathbb E[X\mid\mathcal G].
}
$$


### No information and full information

At the two extremes,

$$
\boxed{
\mathbb E[X\mid\{\varnothing,\Omega\}]
=
\mathbb E[X],
}
$$

while

$$
\boxed{
\mathbb E[X\mid\mathcal F]
=
X.
}
$$

The first retains no random information; the second retains all information.


## 8. Conditional convergence theorems

The ordinary convergence theorems from Chapter 7 have conditional analogues.

### Conditional monotone convergence

If

$$
0\le X_n\uparrow X,
$$

then versions may be chosen so that

$$
\boxed{
\mathbb E[X_n\mid\mathcal G]
\uparrow
\mathbb E[X\mid\mathcal G].
}
$$

### Conditional Fatou

For $X_n\ge0$,

$$
\boxed{
\mathbb E[
\liminf X_n
\mid\mathcal G
]
\le
\liminf
\mathbb E[X_n\mid\mathcal G].
}
$$

### Conditional dominated convergence

If

$$
X_n\to X
$$

almost surely and

$$
|X_n|\le Z\in L^1,
$$

then

$$
\boxed{
\mathbb E[X_n\mid\mathcal G]
\to
\mathbb E[X\mid\mathcal G]
}
$$

almost surely and in $L^1$.


In [ ]:
# Finite-space monotone-convergence illustration.
values_limit = [0, 1, 4, 9, 16, 25]
labels = ["odd" if x%2 else "even" for x in die_values]

rows = []
previous = None

for n in range(1, 6):
    values_n = [min(x, n) for x in values_limit]
    ce_n, means_n = conditional_expectation_partition(
        values_n,
        die_probs,
        labels,
    )

    rows.append((n, float(means_n["odd"]), float(means_n["even"])))

    if previous is not None:
        assert all(a <= b for a,b in zip(previous, ce_n))
    previous = ce_n

display(Markdown(
    "| n | conditional mean on odd cell | conditional mean on even cell |\n"
    "|---:|---:|---:|\n"
    + "\n".join(
        f"| {n} | {odd:.4f} | {even:.4f} |"
        for n, odd, even in rows
    )
))


### $L^1$ continuity

If

$$
X_n\to X
\quad\text{in }L^1,
$$

then

$$
\boxed{
\mathbb E[X_n\mid\mathcal G]
\to
\mathbb E[X\mid\mathcal G]
\quad\text{in }L^1.
}
$$

This follows immediately from linearity and the $L^1$ contraction.


## 9. Testing against known random variables

If $X\in L^1$ and $Z$ is bounded and $\mathcal G$-measurable, then

$$
\boxed{
\mathbb E[ZX]
=
\mathbb E[
Z\mathbb E[X\mid\mathcal G]
].
}
$$

This extends the defining identity from indicators to bounded known random variables.


### Taking out what is known

If $Z$ is $\mathcal G$-measurable, then under the stated non-negative or integrability hypotheses,

$$
\boxed{
\mathbb E[ZX\mid\mathcal G]
=
Z\mathbb E[X\mid\mathcal G].
}
$$

The factor may be pulled out because its value is already known under the retained information.


In [ ]:
# Die example: known factor Z=1_even.
Z = [Fraction(int(x%2==0),1) for x in die_values]
X = [Fraction(x,1) for x in die_values]

EX_given_G, _ = conditional_expectation_partition(
    X,
    die_probs,
    parity,
)

lhs = expectation(
    [z*x for z,x in zip(Z,X)],
    die_probs,
)

rhs = expectation(
    [z*m for z,m in zip(Z,EX_given_G)],
    die_probs,
)

display(Math(r"\mathbb E[ZX]=" + fmt_fraction(lhs)))
display(Math(
    r"\mathbb E[Z\mathbb E[X\mid\mathcal G]]="
    + fmt_fraction(rhs)
))


## 10. Tower property

If

$$
\mathcal H
\subseteq
\mathcal G
\subseteq
\mathcal F,
$$

then for non-negative or integrable $X$,

$$
\boxed{
\mathbb E[
\mathbb E[X\mid\mathcal G]
\mid\mathcal H
]
=
\mathbb E[X\mid\mathcal H].
}
$$

The nesting hypothesis is essential.


### Law of total expectation

Taking

$$
\mathcal H=\{\varnothing,\Omega\}
$$

gives

$$
\boxed{
\mathbb E[X]
=
\mathbb E[
\mathbb E[X\mid\mathcal G]
].
}
$$

For a finite or countable partition $\{A_i\}$,

$$
\boxed{
\mathbb E[X]
=
\sum_i
\mathbb E[X\mid A_i]
P(A_i).
}
$$


### Exact analogy with total probability

Both formulas use the same “split into cases and recombine” architecture:

$$
P(B)
=
\sum_i
P(B\mid A_i)P(A_i),
$$

$$
\boxed{
\mathbb E[X]
=
\sum_i
\mathbb E[X\mid A_i]P(A_i).
}
$$

The unconditional mean is a probability-weighted average of the means inside the information cells.


### Fair die: recombining even and odd means

$$
\mathbb E[X]
=
4\cdot\frac12
+
3\cdot\frac12
=
\frac72.
$$

This is the same mean obtained directly from the six die outcomes.


In [ ]:
direct = expectation(die_values, die_probs)
recombined = Fraction(4,1)*Fraction(1,2) + Fraction(3,1)*Fraction(1,2)

display(Math(r"\mathbb E[X]_{\mathrm{direct}}=" + fmt_fraction(direct)))
display(Math(r"\mathbb E[X]_{\mathrm{partition}}=" + fmt_fraction(recombined)))


### Three risk classes

Suppose

$$
P(A_1)=\frac12,
\qquad
P(A_2)=\frac3{10},
\qquad
P(A_3)=\frac15,
$$

and

$$
\mathbb E[L\mid A_1]=100,
\qquad
\mathbb E[L\mid A_2]=300,
\qquad
\mathbb E[L\mid A_3]=800.
$$

Then

$$
\boxed{
\mathbb E[L]
=
100\cdot\frac12
+
300\cdot\frac3{10}
+
800\cdot\frac15
=
300.
}
$$


In [ ]:
risk_means = [100,300,800]
risk_probs = [Fraction(1,2), Fraction(3,10), Fraction(1,5)]

display(Math(
    r"\mathbb E[L]="
    + fmt_fraction(expectation(risk_means, risk_probs))
))


### Conditioning in two stages with two dice

Let

$$
X=D_1+D_2,
$$

with independent fair dice. Define

$$
\mathcal G=\sigma(D_1)
$$

and let $\mathcal H$ reveal only whether $D_1$ is even.

Then

$$
\mathbb E[X\mid\mathcal G]
=
D_1+\frac72.
$$

Conditioning this again on $\mathcal H$ gives exactly

$$
\mathbb E[X\mid\mathcal H].
$$


## 11. Increasing information: filtrations

A filtration is an increasing family

$$
(\mathcal F_t)
$$

such that

$$
\boxed{
s\le t
\Longrightarrow
\mathcal F_s\subseteq\mathcal F_t.
}
$$

It models information that accumulates over time.


### Adapted processes and natural filtration

A process $(X_t)$ is adapted when

$$
X_t
$$

is $\mathcal F_t$-measurable for every $t$.

Its natural filtration is

$$
\boxed{
\mathcal F_t^X
=
\sigma(X_s:s\le t).
}
$$

In discrete time,

$$
\mathcal F_n^X
=
\sigma(X_0,\ldots,X_n).
$$


### Conditional predictions along a filtration

For fixed $X\in L^1$, define

$$
M_t=\mathbb E[X\mid\mathcal F_t].
$$

Then $M_t$ is adapted and, for $s\le t$,

$$
\boxed{
\mathbb E[M_t\mid\mathcal F_s]
=
M_s.
}
$$

This is the tower property interpreted dynamically and anticipates the martingale viewpoint used later.


In [ ]:
# Four fair coin outcomes for two time steps.
outcomes = [(0,0),(0,1),(1,0),(1,1)]
probs = [Fraction(1,4)]*4
terminal = [a+b for a,b in outcomes]

# F0: no information
F0_labels = ["all"]*4

# F1: first toss known
F1_labels = [f"first={a}" for a,b in outcomes]

# F2: full information
F2_labels = [f"{a}{b}" for a,b in outcomes]

M0, _ = conditional_expectation_partition(terminal, probs, F0_labels)
M1, _ = conditional_expectation_partition(terminal, probs, F1_labels)
M2, _ = conditional_expectation_partition(terminal, probs, F2_labels)

display(Markdown(f"$M_0$: **{M0}**"))
display(Markdown(f"$M_1$: **{M1}**"))
display(Markdown(f"$M_2$: **{M2}**"))

# E[M2 | F1] = M1
tower_check, _ = conditional_expectation_partition(M2, probs, F1_labels)
display(Markdown(f"$E[M_2|F_1]$: **{tower_check}**"))


## 12. Conditioning on independent information

If $X\in L^1$ is independent of $\mathcal G$, then

$$
\boxed{
\mathbb E[X\mid\mathcal G]
=
\mathbb E[X].
}
$$

More generally, if $h(X)\in L^1$,

$$
\mathbb E[h(X)\mid\mathcal G]
=
\mathbb E[h(X)].
$$


### The converse is false

Let $X$ be uniform on

$$
\{-1,0,1\},
$$

let

$$
Y=X^2,
$$

and let

$$
\mathcal G=\sigma(Y).
$$

Then

$$
\boxed{
\mathbb E[X\mid\mathcal G]
=
0
=
\mathbb E[X],
}
$$

but $X$ and $Y$ are not independent because $Y$ is a non-constant function of $X$.


In [ ]:
x_vals = [-1,0,1]
x_probs = [Fraction(1,3)]*3
y_vals = [x*x for x in x_vals]
labels = [f"Y={y}" for y in y_vals]

ce, means = conditional_expectation_partition(
    x_vals,
    x_probs,
    labels,
)

display(Markdown(f"Conditional means by Y-cell: **{means}**"))
display(Math(r"\mathbb E[X]=0"))
display(Markdown("Yet $Y=X^2$ is deterministically determined by $X$."))


## 13. Abstract theory before representations

Up to this point, no conditional density has been needed.

The chapter deliberately completes the abstract operator theory first:

$$
\text{definition}
\to
\text{properties}
\to
\text{convergence}
\to
\text{tower/testing}
\to
\text{Jensen}
\to
\text{variance/covariance}
\to
\text{MMSE}.
$$

Only after this do we introduce formulas based on conditioning on a random variable or on a joint density.


## 14. Conditional Jensen inequality

Let $X\in L^1$, let $\varphi:\mathbb R\to\mathbb R$ be finite and convex, and assume $\varphi(X)\in L^1$.

Then

$$
\boxed{
\varphi(
\mathbb E[X\mid\mathcal G]
)
\le
\mathbb E[
\varphi(X)
\mid\mathcal G
].
}
$$


### Quadratic case

For $X\in L^2$,

$$
\boxed{
\mathbb E[X\mid\mathcal G]^2
\le
\mathbb E[X^2\mid\mathcal G].
}
$$

This guarantees that the conditional mean is also in $L^2$.


In [ ]:
# Jensen on the die parity partition.
ce_X, _ = conditional_expectation_partition(
    die_values,
    die_probs,
    parity,
)

X2 = [x*x for x in die_values]
ce_X2, _ = conditional_expectation_partition(
    X2,
    die_probs,
    parity,
)

jensen_ok = all(
    Fraction(m)**2 <= q
    for m,q in zip(ce_X,ce_X2)
)

display(Markdown(f"Conditional Jensen verified outcome by outcome: **{jensen_ok}**"))
display(Markdown(f"$E[X|G]^2$: **{[m*m for m in ce_X]}**"))
display(Markdown(f"$E[X^2|G]$: **{ce_X2}**"))


### $L^p$ contraction

For

$$
1\le p<\infty
$$

and $X\in L^p$,

$$
\boxed{
\|
\mathbb E[X\mid\mathcal G]
\|_p
\le
\|X\|_p.
}
$$

This follows from conditional Jensen applied to $|x|^p$.


## 15. Conditional variance

For $X\in L^2$,

$$
\boxed{
\operatorname{Var}(X\mid\mathcal G)
=
\mathbb E[
(X-\mathbb E[X\mid\mathcal G])^2
\mid\mathcal G
].
}
$$

The computational formula is

$$
\boxed{
\operatorname{Var}(X\mid\mathcal G)
=
\mathbb E[X^2\mid\mathcal G]
-
\mathbb E[X\mid\mathcal G]^2.
}
$$


### Law of total variance

$$
\boxed{
\operatorname{Var}(X)
=
\mathbb E[
\operatorname{Var}(X\mid\mathcal G)
]
+
\operatorname{Var}(
\mathbb E[X\mid\mathcal G]
).
}
$$

The first term is uncertainty **remaining after conditioning**.

The second is variability **explained by the information**.


### Two independent dice

For

$$
X=D_1+D_2
$$

and

$$
\mathcal G=\sigma(D_1),
$$

$$
\mathbb E[X\mid\mathcal G]
=
D_1+\frac72,
$$

and

$$
\operatorname{Var}(X\mid\mathcal G)
=
\operatorname{Var}(D_2)
=
\frac{35}{12}.
$$

Therefore

$$
\boxed{
\operatorname{Var}(X)
=
\frac{35}{12}
+
\frac{35}{12}
=
\frac{35}{6}.
}
$$


In [ ]:
die_var = Fraction(35,12)
total_var = die_var + die_var

display(Math(
    r"\mathbb E[\operatorname{Var}(X\mid\mathcal G)]="
    + fmt_fraction(die_var)
))
display(Math(
    r"\operatorname{Var}(\mathbb E[X\mid\mathcal G])="
    + fmt_fraction(die_var)
))
display(Math(r"\operatorname{Var}(X)=" + fmt_fraction(total_var)))


In [ ]:
# Numerical experiment from the chapter: total variance with two dice.
tv_N = widgets.IntSlider(
    value=100000,
    min=5000,
    max=500000,
    step=5000,
    description="N",
)
tv_output = widgets.Output()


def update_total_variance_sim(*_):
    with tv_output:
        clear_output(wait=True)

        N = tv_N.value
        rng = np.random.default_rng(2026)

        d1 = rng.integers(1,7,size=N)
        d2 = rng.integers(1,7,size=N)
        x = d1+d2

        conditional_means = np.array([
            x[d1==d].mean()
            for d in range(1,7)
        ])

        conditional_vars = np.array([
            x[d1==d].var()
            for d in range(1,7)
        ])

        first_term = conditional_vars.mean()
        second_term = conditional_means.var()
        total = x.var()

        display(Math(r"\widehat{\operatorname{Var}}(X)=" + f"{total:.6f}"))
        display(Math(
            r"\widehat{\mathbb E[\operatorname{Var}(X\mid D_1)]}="
            + f"{first_term:.6f}"
        ))
        display(Math(
            r"\widehat{\operatorname{Var}(\mathbb E[X\mid D_1])}="
            + f"{second_term:.6f}"
        ))
        display(Math(
            r"\text{sum of components}="
            + f"{first_term+second_term:.6f}"
        ))


tv_N.observe(update_total_variance_sim, names="value")
display(widgets.VBox([tv_N, tv_output]))
update_total_variance_sim()


## 16. Conditional covariance

For $X,Y\in L^2$,

$$
\boxed{
\operatorname{Cov}(X,Y\mid\mathcal G)
=
\mathbb E[
(X-\mathbb E[X\mid\mathcal G])
(Y-\mathbb E[Y\mid\mathcal G])
\mid\mathcal G
].
}
$$

Equivalently,

$$
\boxed{
\operatorname{Cov}(X,Y\mid\mathcal G)
=
\mathbb E[XY\mid\mathcal G]
-
\mathbb E[X\mid\mathcal G]
\mathbb E[Y\mid\mathcal G].
}
$$


### Law of total covariance

$$
\boxed{
\operatorname{Cov}(X,Y)
=
\mathbb E[
\operatorname{Cov}(X,Y\mid\mathcal G)
]
+
\operatorname{Cov}(
\mathbb E[X\mid\mathcal G],
\mathbb E[Y\mid\mathcal G]
).
}
$$


### Shared-signal model

Let

$$
X=Z+\varepsilon_1,
\qquad
Y=Z+\varepsilon_2,
$$

where $Z,\varepsilon_1,\varepsilon_2$ are independent, centered and square-integrable.

With

$$
\mathcal G=\sigma(Z),
$$

$$
\mathbb E[X\mid\mathcal G]
=
\mathbb E[Y\mid\mathcal G]
=
Z,
$$

and

$$
\operatorname{Cov}(X,Y\mid\mathcal G)=0.
$$

Hence

$$
\boxed{
\operatorname{Cov}(X,Y)
=
\operatorname{Var}(Z).
}
$$


In [ ]:
signal_var = widgets.FloatSlider(
    value=4.0, min=0.1, max=10.0, step=0.1, description="Var(Z)"
)
noise1_var = widgets.FloatSlider(
    value=1.0, min=0.1, max=10.0, step=0.1, description="Var(e1)"
)
noise2_var = widgets.FloatSlider(
    value=2.0, min=0.1, max=10.0, step=0.1, description="Var(e2)"
)
shared_output = widgets.Output()


def update_shared_signal(*_):
    with shared_output:
        clear_output(wait=True)

        vz = signal_var.value
        v1 = noise1_var.value
        v2 = noise2_var.value

        cov = vz
        corr = cov / math.sqrt((vz+v1)*(vz+v2))

        display(Math(r"\operatorname{Cov}(X,Y)=" + f"{cov:.6f}"))
        display(Math(r"\rho_{X,Y}=" + f"{corr:.6f}"))


for control in (signal_var, noise1_var, noise2_var):
    control.observe(update_shared_signal, names="value")

display(widgets.VBox([
    signal_var,
    noise1_var,
    noise2_var,
    shared_output,
]))
update_shared_signal()


## 17. Conditional expectation as the best mean-square predictor

Let

$$
M=\mathbb E[X\mid\mathcal G]
$$

with $X\in L^2$.

For every $\mathcal G$-measurable $Z\in L^2$,

$$
\boxed{
\mathbb E[(X-Z)^2]
=
\mathbb E[(X-M)^2]
+
\mathbb E[(M-Z)^2].
}
$$

Therefore $M$ uniquely minimizes mean squared prediction error, up to almost-sure equality.


### Orthogonality

The prediction error satisfies

$$
\boxed{
\mathbb E[
(X-M)Z
]
=
0
}
$$

for every square-integrable $\mathcal G$-measurable $Z$.

This is the projection geometry behind conditional expectation in $L^2$.


### More information cannot worsen MMSE

If

$$
\mathcal H\subseteq\mathcal G,
$$

then

$$
\boxed{
\mathbb E[
(X-\mathbb E[X\mid\mathcal G])^2
]
\le
\mathbb E[
(X-\mathbb E[X\mid\mathcal H])^2
].
}
$$

More information enlarges the class of admissible predictors.


In [ ]:
# Exact MMSE comparison for the two-dice sum.
outcomes = [(d1,d2) for d1 in range(1,7) for d2 in range(1,7)]
probs = [Fraction(1,36)]*36
targets = [d1+d2 for d1,d2 in outcomes]

# No information: best constant predictor 7.
pred_none = [Fraction(7,1)]*36

# Observe D1: best predictor D1+7/2.
pred_d1 = [Fraction(d1,1)+Fraction(7,2) for d1,d2 in outcomes]

mse_none = weighted_mse(targets,pred_none,probs)
mse_d1 = weighted_mse(targets,pred_d1,probs)

display(Math(r"\mathrm{MSE}_{\mathrm{no\ info}}=" + fmt_fraction(mse_none)))
display(Math(r"\mathrm{MSE}_{D_1\ \mathrm{known}}=" + fmt_fraction(mse_d1)))
display(Markdown(f"More information improves prediction: **{mse_d1 <= mse_none}**"))


## 18. Conditioning on a random variable

The notation

$$
\boxed{
\mathbb E[X\mid Y]
:=
\mathbb E[X\mid\sigma(Y)]
}
$$

means that the retained information is exactly the information generated by $Y$.


### Doob--Dynkin representation

Because

$$
\mathbb E[X\mid Y]
$$

is $\sigma(Y)$-measurable, there exists a Borel function $m$ such that

$$
\boxed{
\mathbb E[X\mid Y]
=
m(Y)
\quad\text{a.s.}
}
$$

The function $m$ is determined only $P_Y$-almost everywhere.


### Conditioning on a particular value

Writing

$$
\mathbb E[X\mid Y=y]
=
m(y)
$$

requires choosing a particular Borel version $m$.

If $Y$ is continuous, the event $\{Y=y\}$ may have probability zero, so the elementary quotient

$$
\frac{
\mathbb E[X\mathbf 1_{\{Y=y\}}]
}{
P(Y=y)
}
$$

is not available.


## 19. Countably valued conditioning variable

If $Y$ takes countably many values and

$$
P(Y=y)>0,
$$

define

$$
\boxed{
m(y)
=
\frac{
\mathbb E[X\mathbf 1_{\{Y=y\}}]
}{
P(Y=y)
}.
}
$$

Then

$$
\boxed{
\mathbb E[X\mid Y]
=
m(Y).
}
$$


### Two dice: predicting the sum from the first die

For independent fair dice,

$$
X=D_1+D_2.
$$

Then

$$
\boxed{
\mathbb E[X\mid D_1]
=
D_1+\frac72.
}
$$


In [ ]:
two_dice_outcomes = [(d1,d2) for d1 in range(1,7) for d2 in range(1,7)]
two_dice_probs = [Fraction(1,36)]*36
two_dice_sum = [d1+d2 for d1,d2 in two_dice_outcomes]
d1_labels = [f"D1={d1}" for d1,d2 in two_dice_outcomes]

_, d1_means = conditional_expectation_partition(
    two_dice_sum,
    two_dice_probs,
    d1_labels,
)

display(Markdown(
    "| d | E[D1+D2 | D1=d] |\n"
    "|---:|---:|\n"
    + "\n".join(
        f"| {d} | {float(d1_means[f'D1={d}']):.1f} |"
        for d in range(1,7)
    )
))


## 20. Conditional densities

Suppose $(X,Y)$ has joint density $f_{X,Y}$ and

$$
f_Y(y)
=
\int_{\mathbb R}
f_{X,Y}(x,y)\,dx.
$$

For

$$
0<f_Y(y)<\infty,
$$

define

$$
\boxed{
f_{X\mid Y}(x\mid y)
=
\frac{
f_{X,Y}(x,y)
}{
f_Y(y)
}.
}
$$

This is a representation of the abstract conditional law, not the general definition of conditional expectation.


### Conditional mean from a joint density

Under integrability,

$$
\boxed{
\mathbb E[X\mid Y=y]
=
\int_{\mathbb R}
x f_{X\mid Y}(x\mid y)\,dx
}
$$

for an appropriate version and for $P_Y$-almost every $y$.

Then

$$
\mathbb E[X\mid Y]
=
m(Y)
$$

with

$$
m(y)
=
\int x f_{X\mid Y}(x\mid y)\,dx.
$$


### Uniform conditional law on a triangle

Suppose

$$
f_{X,Y}(x,y)
=
2\mathbf 1_{\{0<x<y<1\}}.
$$

For $0<y<1$,

$$
f_Y(y)
=
2y,
$$

so

$$
f_{X\mid Y}(x\mid y)
=
\frac1y
\mathbf 1_{(0,y)}(x).
$$

Thus

$$
X\mid(Y=y)
\sim
U(0,y),
$$

and

$$
\boxed{
\mathbb E[X\mid Y=y]
=
\frac y2,
\qquad
\mathbb E[X\mid Y]
=
\frac Y2.
}
$$


In [ ]:
tri_y = widgets.FloatSlider(
    value=0.7, min=0.05, max=0.95, step=0.05, description="y"
)
tri_output = widgets.Output()


def update_triangle_conditional(*_):
    with tri_output:
        clear_output(wait=True)

        y = tri_y.value
        x = np.linspace(0,y,500)
        dens = np.full_like(x,1/y)

        fig, ax = plt.subplots(figsize=(8,3.0))
        ax.plot(x,dens)
        ax.fill_between(x,0,dens,alpha=0.2)
        ax.set_xlabel("x")
        ax.set_ylabel("conditional density")
        ax.set_title("X | Y=y is uniform on (0,y)")
        plt.show()

        display(Math(
            r"\mathbb E[X\mid Y=y]=" + f"{y/2:.6f}"
        ))


tri_y.observe(update_triangle_conditional, names="value")
display(widgets.VBox([tri_y,tri_output]))
update_triangle_conditional()


### Density form of total expectation

For the triangular model,

$$
m(y)=\frac y2,
\qquad
f_Y(y)=2y,
$$

so

$$
\mathbb E[X]
=
\int_0^1
\frac y2
(2y)\,dy
=
\frac13.
$$

This is the density representation of the abstract law of total expectation.


### A different conditional density

Consider

$$
f(x,y)
=
e^{-x}
\mathbf 1_{\{0<y<x\}}.
$$

Normalization follows from

$$
\int_0^\infty
\int_0^x
e^{-x}\,dy\,dx
=
\int_0^\infty
xe^{-x}\,dx
=
1.
$$

The marginal density of $X$ is

$$
f_X(x)
=
xe^{-x},
\qquad
x>0.
$$

Therefore

$$
\boxed{
f_{Y\mid X}(y\mid x)
=
\frac1x
\mathbf 1_{(0,x)}(y),
}
$$

and

$$
\boxed{
\mathbb E[Y\mid X]
=
\frac X2.
}
$$


In [ ]:
cond_x = widgets.FloatSlider(
    value=2.0, min=0.2, max=6.0, step=0.1, description="x"
)
cond_density_output = widgets.Output()


def update_other_density(*_):
    with cond_density_output:
        clear_output(wait=True)

        x0 = cond_x.value
        y = np.linspace(0,x0,500)
        density = np.full_like(y,1/x0)

        fig, ax = plt.subplots(figsize=(8,3.0))
        ax.plot(y,density)
        ax.set_xlabel("y")
        ax.set_ylabel("f_{Y|X}(y|x)")
        ax.set_title("Conditional uniform law on (0,x)")
        plt.show()

        display(Math(
            r"\mathbb E[Y\mid X=x]=" + f"{x0/2:.6f}"
        ))


cond_x.observe(update_other_density, names="value")
display(widgets.VBox([cond_x,cond_density_output]))
update_other_density()


## 21. Conditional law of a bivariate normal pair

Let $(X,Y)$ be a non-degenerate bivariate normal pair with means $\mu_X,\mu_Y$, standard deviations $\sigma_X,\sigma_Y>0$, and correlation $\rho\in(-1,1)$.

Then

$$
\boxed{
X\mid(Y=y)
\sim
N\left(
\mu_X
+
\rho
\frac{\sigma_X}{\sigma_Y}
(y-\mu_Y),
\;
\sigma_X^2(1-\rho^2)
\right).
}
$$

Consequently,

$$
\boxed{
\mathbb E[X\mid Y]
=
\mu_X
+
\rho
\frac{\sigma_X}{\sigma_Y}
(Y-\mu_Y).
}
$$


### Conditional variance of a bivariate normal pair

The conditional variance is constant:

$$
\boxed{
\operatorname{Var}(X\mid Y)
=
\sigma_X^2(1-\rho^2).
}
$$

The observed value of $Y$ shifts the conditional mean but does not change this conditional variance parameter.


In [ ]:
bn_mu_x = widgets.FloatSlider(value=1.0, min=-5, max=5, step=0.25, description="mu_X")
bn_mu_y = widgets.FloatSlider(value=0.0, min=-5, max=5, step=0.25, description="mu_Y")
bn_sigma_x = widgets.FloatSlider(value=2.0, min=0.2, max=5, step=0.2, description="sigma_X")
bn_sigma_y = widgets.FloatSlider(value=1.5, min=0.2, max=5, step=0.2, description="sigma_Y")
bn_rho = widgets.FloatSlider(value=0.6, min=-0.95, max=0.95, step=0.05, description="rho")
bn_y = widgets.FloatSlider(value=1.0, min=-5, max=5, step=0.25, description="y")
bn_output = widgets.Output()


def update_bvn_conditional(*_):
    with bn_output:
        clear_output(wait=True)

        mu_x = bn_mu_x.value
        mu_y = bn_mu_y.value
        sx = bn_sigma_x.value
        sy = bn_sigma_y.value
        rho = bn_rho.value
        y0 = bn_y.value

        cm = bivariate_normal_conditional_mean(
            y0, mu_x, mu_y, sx, sy, rho
        )
        cv = bivariate_normal_conditional_variance(sx, rho)
        csd = math.sqrt(cv)

        display(Math(
            r"\mathbb E[X\mid Y=y]=" + f"{cm:.6f}"
        ))
        display(Math(
            r"\operatorname{Var}(X\mid Y)=" + f"{cv:.6f}"
        ))

        grid = np.linspace(cm-4*csd, cm+4*csd, 900)
        dens = normal_pdf(grid, cm, csd)

        fig, ax = plt.subplots(figsize=(8,3.3))
        ax.plot(grid,dens)
        ax.axvline(cm,linestyle="--")
        ax.set_xlabel("x")
        ax.set_ylabel("conditional density")
        ax.set_title("Bivariate-normal conditional distribution")
        plt.show()


for control in (
    bn_mu_x,bn_mu_y,bn_sigma_x,bn_sigma_y,bn_rho,bn_y
):
    control.observe(update_bvn_conditional, names="value")

display(widgets.VBox([
    widgets.HBox([bn_mu_x,bn_mu_y]),
    widgets.HBox([bn_sigma_x,bn_sigma_y]),
    widgets.HBox([bn_rho,bn_y]),
    bn_output,
]))
update_bvn_conditional()


## 22. Historical problem: Galton and regression toward the mean

Let $H$ be a standardized parental measurement and $C$ a standardized offspring measurement.

Assume a bivariate normal model with

$$
\mathbb E[H]
=
\mathbb E[C]
=
0,
$$

$$
\operatorname{Var}(H)
=
\operatorname{Var}(C)
=
1,
$$

and

$$
0<\rho<1.
$$

Then

$$
\boxed{
\mathbb E[C\mid H]
=
\rho H.
}
$$

For $h>0$,

$$
0<\rho h<h.
$$

For $h<0$, the conditional mean again lies between $h$ and the population mean $0$.

“Regression toward the mean” describes the conditional mean, not the path of every individual observation.


In [ ]:
galton_rho = widgets.FloatSlider(
    value=0.6, min=0.05, max=0.95, step=0.05, description="rho"
)
galton_output = widgets.Output()


def update_galton_regression(*_):
    with galton_output:
        clear_output(wait=True)

        rho = galton_rho.value
        h = np.linspace(-3,3,300)

        fig, ax = plt.subplots(figsize=(6,5))
        ax.plot(h,h,linestyle="--",label="c=h")
        ax.plot(h,rho*h,label="E[C|H=h]=rho h")
        ax.axhline(0,linewidth=0.8)
        ax.axvline(0,linewidth=0.8)
        ax.set_xlabel("parental standardized value h")
        ax.set_ylabel("offspring standardized value c")
        ax.set_title("Regression toward the population mean")
        ax.legend()
        plt.show()


galton_rho.observe(update_galton_regression, names="value")
display(widgets.VBox([galton_rho,galton_output]))
update_galton_regression()


## 23. Conditioning does not repair model misspecification

Conditional expectation is mathematically meaningful once a probability model has been specified.

It does **not** prove that the model describes the real data-generating mechanism.

A perfectly computed

$$
\mathbb E[X\mid\mathcal G]
$$

may still be practically misleading if the assumed model is wrong, unstable through time, or missing relevant state variables.

This is especially important when using conditional expectations for forecasting or financial decisions.


## 24. Python laboratory: estimating conditional means by grouping

For discrete conditioning variables, conditional means can be estimated empirically by grouping observations according to the observed information.

For the two-dice model,

$$
\mathbb E[D_1+D_2\mid D_1=d]
=
d+\frac72.
$$


In [ ]:
group_N = widgets.IntSlider(
    value=100000,
    min=5000,
    max=500000,
    step=5000,
    description="N",
)
group_output = widgets.Output()


def update_grouping(*_):
    with group_output:
        clear_output(wait=True)

        N = group_N.value
        rng = np.random.default_rng(2026)

        d1 = rng.integers(1,7,size=N)
        d2 = rng.integers(1,7,size=N)
        x = d1+d2

        rows = []
        for d in range(1,7):
            estimate = x[d1==d].mean()
            theory = d+3.5
            rows.append(
                f"| {d} | {estimate:.4f} | {theory:.1f} |"
            )

        display(Markdown(
            "| d | estimated E[X|D1=d] | theory |\n"
            "|---:|---:|---:|\n"
            + "\n".join(rows)
        ))


group_N.observe(update_grouping, names="value")
display(widgets.VBox([group_N,group_output]))
update_grouping()


## 25. AI Audit example: when the tower property fails

An incorrect claim is

$$
\mathbb E[
\mathbb E[X\mid\mathcal G]
\mid\mathcal H
]
=
\mathbb E[X\mid\mathcal H]
$$

for arbitrary $\mathcal G$ and $\mathcal H$.

The tower theorem requires nested information.


### Explicit counterexample

Let $U,V$ be independent Bernoulli$(1/2)$ variables and define

$$
X=UV,
$$

$$
\mathcal G=\sigma(U),
\qquad
\mathcal H=\sigma(V).
$$

Then

$$
\mathbb E[X\mid\mathcal G]
=
\frac U2.
$$

Since $U$ is independent of $\mathcal H$,

$$
\mathbb E[
\mathbb E[X\mid\mathcal G]
\mid\mathcal H
]
=
\frac14.
$$

But

$$
\mathbb E[X\mid\mathcal H]
=
\frac V2.
$$

Therefore the two sides are different.


In [ ]:
uv_outcomes = [(0,0),(0,1),(1,0),(1,1)]
uv_probs = [Fraction(1,4)]*4
Xuv = [u*v for u,v in uv_outcomes]

G_labels = [f"U={u}" for u,v in uv_outcomes]
H_labels = [f"V={v}" for u,v in uv_outcomes]

EX_G, _ = conditional_expectation_partition(
    Xuv,
    uv_probs,
    G_labels,
)

tower_wrong_left, _ = conditional_expectation_partition(
    EX_G,
    uv_probs,
    H_labels,
)

EX_H, _ = conditional_expectation_partition(
    Xuv,
    uv_probs,
    H_labels,
)

display(Markdown(f"$E[X|G]$: **{EX_G}**"))
display(Markdown(f"$E[E[X|G]|H]$: **{tower_wrong_left}**"))
display(Markdown(f"$E[X|H]$: **{EX_H}**"))
display(Markdown(
    f"Equal? **{tower_wrong_left == EX_H}**"
))


## 26. Guided exercise generator


In [ ]:
exercise_rng = random.Random(20260815)

exercise_kind = widgets.Dropdown(
    options=[
        ("Random", "random"),
        ("Event conditioning", "event"),
        ("Partition", "partition"),
        ("Tower", "tower"),
        ("Independent information", "independence"),
        ("Total expectation", "total"),
        ("Conditional variance", "variance"),
        ("MMSE", "mmse"),
        ("Conditional density", "density"),
        ("Bivariate normal", "normal"),
    ],
    value="random",
    description="Type",
)

new_button = widgets.Button(description="New exercise")
hint_button = widgets.Button(description="Hint")
reveal_button = widgets.Button(description="Reveal")
check_button = widgets.Button(description="Check")
answer_box = widgets.Text(description="Answer")
prompt_output = widgets.Output()
feedback_output = widgets.Output()
state = {}


def make_exercise(_=None):
    kind = exercise_kind.value

    if kind == "random":
        kind = exercise_rng.choice([
            "event",
            "partition",
            "tower",
            "independence",
            "total",
            "variance",
            "mmse",
            "density",
            "normal",
        ])

    if kind == "event":
        target = "4"
        prompt = "A fair die is observed to be even. Find E[X | even]."
        hint = "Average 2,4,6."
        solution = r"\mathbb E[X\mid X\text{ even}]=4."

    elif kind == "partition":
        target = "3.5"
        prompt = "Using even/odd information for a fair die, recombine the conditional means to find E[X]."
        hint = "Use 4(1/2)+3(1/2)."
        solution = r"\mathbb E[X]=7/2."

    elif kind == "tower":
        target = "no"
        prompt = "Does the tower identity hold for arbitrary unrelated sigma-algebras G and H? yes/no"
        hint = "The theorem requires one sigma-algebra to be contained in the other."
        solution = r"\text{No.}"

    elif kind == "independence":
        target = "3.5"
        prompt = "D2 is a fair die independent of sigma(D1). Find E[D2 | sigma(D1)]."
        hint = "Independent information does not change the mean."
        solution = r"\mathbb E[D_2\mid\sigma(D_1)]=7/2."

    elif kind == "total":
        target = "300"
        prompt = "Risk-class probabilities are 0.5,0.3,0.2 with conditional means 100,300,800. Find the total mean."
        hint = "Take the probability-weighted average."
        solution = r"\mathbb E[L]=300."

    elif kind == "variance":
        target = str(35/6)
        prompt = "For X=D1+D2 with independent fair dice, find Var(X)."
        hint = "Each die has variance 35/12."
        solution = r"\operatorname{Var}(X)=35/6."

    elif kind == "mmse":
        target = "yes"
        prompt = "Is E[X|G] the unique G-measurable L2 predictor minimizing mean squared error, up to a.s. equality? yes/no"
        hint = "Use the Pythagorean MMSE identity."
        solution = r"\text{Yes.}"

    elif kind == "density":
        target = "0.35"
        prompt = "In the triangular conditional model X|Y=y ~ U(0,y), find E[X|Y=0.7]."
        hint = "The mean of U(0,y) is y/2."
        solution = r"\mathbb E[X\mid Y=0.7]=0.35."

    else:
        target = "0.64"
        prompt = "In a standardized bivariate normal pair with rho=0.6 and Var(X)=1, find Var(X|Y)."
        hint = "Use 1-rho^2."
        solution = r"\operatorname{Var}(X\mid Y)=0.64."

    state.clear()
    state.update(
        target=target,
        hint=hint,
        solution=solution,
    )

    answer_box.value = ""

    with prompt_output:
        clear_output(wait=True)
        display(Markdown("### Exercise\n" + prompt))

    with feedback_output:
        clear_output(wait=True)


def show_hint(_):
    with feedback_output:
        clear_output(wait=True)
        display(Markdown("**Hint:** " + state["hint"]))


def reveal(_):
    with feedback_output:
        clear_output(wait=True)
        display(Math(state["solution"]))


def check(_):
    with feedback_output:
        clear_output(wait=True)

        guess = answer_box.value.strip().lower().replace(" ","")
        target = state["target"].replace(" ","")

        correct = guess == target

        if not correct:
            try:
                correct = abs(float(guess)-float(target)) < 5e-4
            except Exception:
                pass

        display(Markdown(
            "**Correct.**"
            if correct
            else "**Not yet. Identify the retained information before computing.**"
        ))


new_button.on_click(make_exercise)
hint_button.on_click(show_hint)
reveal_button.on_click(reveal)
check_button.on_click(check)

display(widgets.VBox([
    widgets.HBox([exercise_kind,new_button]),
    prompt_output,
    widgets.HBox([answer_box,check_button]),
    widgets.HBox([hint_button,reveal_button]),
    feedback_output,
]))

make_exercise()


## 27. AI Audit: conditional expectation

Use this checklist on any AI-generated solution.

1. Is the conditioning object an event, a random variable, or a $\sigma$-algebra?
2. If conditioning on an event by quotient, is its probability positive?
3. Is $\mathbb E[X\mid\mathcal G]$ treated as a random variable rather than automatically as a scalar?
4. Is the result $\mathcal G$-measurable?
5. Does it satisfy the defining identity against every $A\in\mathcal G$?
6. Are two versions allowed to differ on a null set?
7. Is conditional probability recognized as $\mathbb E[\mathbf 1_B\mid\mathcal G]$?
8. Is mean preservation used correctly?
9. Is a known $\mathcal G$-measurable quantity pulled out only under the correct hypotheses?
10. Does a tower-property argument verify $\mathcal H\subseteq\mathcal G$?
11. Is the law of total expectation used as a weighted recombination over a genuine partition?
12. Is a filtration actually increasing?
13. Is an adapted process measurable with respect to current information?
14. Is independence of a random variable from $\mathcal G$ stated in terms of $\sigma(X)$?
15. Is the false converse “constant conditional mean implies independence” avoided?
16. Are conditional MCT, Fatou and DCT used with their actual hypotheses?
17. Does conditional Jensen use a convex function and the required integrability?
18. Is conditional variance computed as $E[X^2|G]-E[X|G]^2$?
19. Is total variance interpreted as remaining plus explained uncertainty?
20. Is total covariance split into residual conditional covariance plus covariance of conditional means?
21. Is the MMSE predictor restricted to $\mathcal G$-measurable competitors?
22. Is conditioning on $Y$ understood as conditioning on $\sigma(Y)$?
23. Is a pointwise value $E[X|Y=y]$ tied to a chosen version?
24. Is the density ratio used only when a joint density exists and $f_Y(y)>0$?
25. Is the conditional-density formula treated as a representation rather than the general definition?
26. Is the bivariate normal conditional variance $\sigma_X^2(1-\rho^2)$?
27. Is Galton's “regression toward the mean” interpreted as a statement about a conditional mean, not every individual observation?
28. Is a correct conditional expectation being confused with evidence that the underlying model is empirically correct?
29. Is numerical simulation used as illustration rather than proof?

### Claims to audit

- “Conditional expectation given information is always a number.”
- “The tower property holds for any two sigma-algebras.”
- “If $E[X|G]=E[X]$, then $X$ is independent of $G$.”
- “For continuous $Y$, define $E[X|Y=y]$ by dividing by $P(Y=y)$.”
- “A good conditional forecast proves that the probability model is correctly specified.”

All five claims are false in general.


### Suggested AI-guided activities

- “Give me a finite partition and make me construct the conditional expectation cell by cell, then verify the defining identity.”
- “Give me a tower-property claim and make me check sigma-algebra nesting before doing any computation.”
- “Construct a constant-conditional-mean example that is still dependent.”
- “Ask me to decompose total variance into remaining and explained uncertainty.”
- “Give me a shared-signal model and make me identify the two terms in total covariance.”
- “Give me three candidate predictors based on the same information and make me verify the MMSE Pythagorean identity.”
- “Give me a joint density and make me derive the marginal density before writing the conditional density.”
- “Ask me to distinguish the abstract conditional expectation from a chosen version of $E[X|Y=y]$.”


## 28. Self-check quiz


In [ ]:
quiz_data = [
    (
        "1. E[X|G] is generally:",
        ["Choose...", "a G-measurable random variable", "always a scalar"],
        "a G-measurable random variable",
        r"\mathbb E[X\mid\mathcal G]\text{ depends only on the retained information.}",
    ),
    (
        "2. The abstract defining identity is:",
        [
            "Choose...",
            "E[E[X|G] 1_A]=E[X 1_A] for all A in G",
            "E[X|G]=E[X] always",
        ],
        "E[E[X|G] 1_A]=E[X 1_A] for all A in G",
        r"\mathbb E[\mathbb E[X\mid\mathcal G]\mathbf1_A]=\mathbb E[X\mathbf1_A].",
    ),
    (
        "3. Conditional expectation is unique:",
        ["Choose...", "pointwise", "up to almost-sure equality"],
        "up to almost-sure equality",
        r"\text{Different versions may differ on a null set.}",
    ),
    (
        "4. Conditional probability P(B|G) equals:",
        ["Choose...", "E[1_B|G]", "P(B)/P(G)"],
        "E[1_B|G]",
        r"P(B\mid\mathcal G)=\mathbb E[\mathbf1_B\mid\mathcal G].",
    ),
    (
        "5. The tower property requires:",
        ["Choose...", "nested sigma-algebras", "independence of sigma-algebras"],
        "nested sigma-algebras",
        r"\mathcal H\subseteq\mathcal G.",
    ),
    (
        "6. Independent information gives:",
        [
            "Choose...",
            "E[X|G]=E[X]",
            "E[X|G]=X",
        ],
        "E[X|G]=E[X]",
        r"X\perp\mathcal G\Longrightarrow\mathbb E[X\mid\mathcal G]=\mathbb E[X].",
    ),
    (
        "7. Constant conditional mean implies independence in general:",
        ["Choose...", "true", "false"],
        "false",
        r"\text{Use }Y=X^2\text{ with symmetric }X.",
    ),
    (
        "8. Conditional Jensen for convex phi says:",
        [
            "Choose...",
            "phi(E[X|G]) <= E[phi(X)|G]",
            "phi(E[X|G]) = E[phi(X)|G] always",
        ],
        "phi(E[X|G]) <= E[phi(X)|G]",
        r"\varphi(\mathbb E[X\mid\mathcal G])\le\mathbb E[\varphi(X)\mid\mathcal G].",
    ),
    (
        "9. Total variance decomposes into:",
        [
            "Choose...",
            "remaining conditional variance plus variance of conditional mean",
            "two equal terms always",
        ],
        "remaining conditional variance plus variance of conditional mean",
        r"\operatorname{Var}(X)=\mathbb E[\operatorname{Var}(X\mid\mathcal G)]+\operatorname{Var}(\mathbb E[X\mid\mathcal G]).",
    ),
    (
        "10. E[X|Y] means:",
        ["Choose...", "E[X|sigma(Y)]", "E[X|Y=y] for one arbitrary y"],
        "E[X|sigma(Y)]",
        r"\mathbb E[X\mid Y]=\mathbb E[X\mid\sigma(Y)].",
    ),
    (
        "11. A conditional density is:",
        [
            "Choose...",
            "a representation available under a joint-density assumption",
            "the general definition of conditional expectation",
        ],
        "a representation available under a joint-density assumption",
        r"\text{The abstract theory comes first.}",
    ),
    (
        "12. In the bivariate normal family, Var(X|Y) equals:",
        [
            "Choose...",
            "sigma_X^2(1-rho^2)",
            "sigma_X^2 rho^2",
            "sigma_X^2",
        ],
        "sigma_X^2(1-rho^2)",
        r"\operatorname{Var}(X\mid Y)=\sigma_X^2(1-\rho^2).",
    ),
]

quiz_widgets = []
quiz_rows = []

for prompt, options, _, _ in quiz_data:
    dropdown = widgets.Dropdown(
        options=options,
        value="Choose...",
        layout=widgets.Layout(width="500px"),
    )
    quiz_widgets.append(dropdown)
    quiz_rows.append(widgets.HBox([
        widgets.HTML(f"<div style='width:670px'>{prompt}</div>"),
        dropdown,
    ]))

grade_button = widgets.Button(description="Grade quiz")
quiz_output = widgets.Output()


def grade_quiz(_):
    with quiz_output:
        clear_output(wait=True)

        score = sum(
            widget.value == correct
            for widget, (_,_,correct,_) in zip(
                quiz_widgets,
                quiz_data,
            )
        )

        display(Markdown(f"### Score: {score}/{len(quiz_data)}"))

        for i, (
            widget,
            (_,_,correct,explanation),
        ) in enumerate(zip(quiz_widgets,quiz_data),1):

            mark = "✓" if widget.value == correct else "✗"

            display(Markdown(
                f"**{mark} Question {i}:** correct answer = `{correct}`"
            ))

            display(Math(explanation))


grade_button.on_click(grade_quiz)

display(widgets.VBox(
    quiz_rows + [grade_button, quiz_output]
))


## 29. Automatic mathematical verification

The final code cell checks representative identities from the chapter.


In [ ]:
# Fair die conditional expectation by parity.
values = [1,2,3,4,5,6]
probs = [Fraction(1,6)]*6
labels = ["odd" if x%2 else "even" for x in values]

ce, means = conditional_expectation_partition(values,probs,labels)

assert means["odd"] == 3
assert means["even"] == 4
assert expectation(ce,probs) == Fraction(7,2)

# Event conditioning.
even_mask = [x%2==0 for x in values]
assert conditional_mean_on_event(values,probs,even_mask) == 4

# Testing identity with Z=indicator even.
Z = [Fraction(int(x%2==0),1) for x in values]

lhs = expectation(
    [z*x for z,x in zip(Z,values)],
    probs,
)
rhs = expectation(
    [z*m for z,m in zip(Z,ce)],
    probs,
)

assert lhs == rhs

# Conditional Jensen for square.
ce_x2, _ = conditional_expectation_partition(
    [x*x for x in values],
    probs,
    labels,
)

assert all(
    Fraction(m)**2 <= q
    for m,q in zip(ce,ce_x2)
)

# Two-dice total expectation and variance.
outcomes = [(d1,d2) for d1 in range(1,7) for d2 in range(1,7)]
p36 = [Fraction(1,36)]*36
total = [d1+d2 for d1,d2 in outcomes]

assert expectation(total,p36) == 7

mean_total = Fraction(7,1)
var_total = sum(
    p*(Fraction(x)-mean_total)**2
    for x,p in zip(total,p36)
)

assert var_total == Fraction(35,6)

d1_labels = [f"D1={d1}" for d1,d2 in outcomes]
ce_sum, _ = conditional_expectation_partition(total,p36,d1_labels)

expected_ce = [
    Fraction(d1,1)+Fraction(7,2)
    for d1,d2 in outcomes
]

assert ce_sum == expected_ce

# MMSE: more information lowers error.
pred_none = [Fraction(7,1)]*36
mse_none = weighted_mse(total,pred_none,p36)
mse_d1 = weighted_mse(total,ce_sum,p36)

assert mse_d1 == Fraction(35,12)
assert mse_none == Fraction(35,6)
assert mse_d1 <= mse_none

# Tower failure when information is not nested.
uv_outcomes = [(0,0),(0,1),(1,0),(1,1)]
uv_probs = [Fraction(1,4)]*4
Xuv = [u*v for u,v in uv_outcomes]
G = [f"U={u}" for u,v in uv_outcomes]
H = [f"V={v}" for u,v in uv_outcomes]

EXG, _ = conditional_expectation_partition(Xuv,uv_probs,G)
left, _ = conditional_expectation_partition(EXG,uv_probs,H)
EXH, _ = conditional_expectation_partition(Xuv,uv_probs,H)

assert left != EXH
assert left == [Fraction(1,4)]*4
assert EXH == [
    Fraction(0,1),
    Fraction(1,2),
    Fraction(0,1),
    Fraction(1,2),
]

# Constant conditional mean without independence.
x = [-1,0,1]
p = [Fraction(1,3)]*3
Y = [1,0,1]
Ylabels = [f"Y={y}" for y in Y]

ce_sym, _ = conditional_expectation_partition(x,p,Ylabels)

assert ce_sym == [0,0,0]

# Total covariance shared-signal identity.
vz = 4.0
v1 = 1.0
v2 = 2.0

assert abs(vz - 4.0) < 1e-12
assert 0 < vz/math.sqrt((vz+v1)*(vz+v2)) < 1

# Triangular conditional law.
y = 0.7
assert abs(y/2 - 0.35) < 1e-12

# Other conditional density.
x0 = 2.0
assert abs(x0/2 - 1.0) < 1e-12

# Bivariate normal conditional formulas.
mu_x,mu_y = 1.0,0.0
sx,sy,rho = 2.0,1.5,0.6
y0 = 1.0

cm = bivariate_normal_conditional_mean(y0,mu_x,mu_y,sx,sy,rho)
cv = bivariate_normal_conditional_variance(sx,rho)

assert abs(cm - 1.8) < 1e-12
assert abs(cv - 2.56) < 1e-12

# Galton shrinkage.
rho = 0.6

for h in [-2,-1,1,2]:
    prediction = rho*h
    if h > 0:
        assert 0 < prediction < h
    else:
        assert h < prediction < 0

show_result(
    "All Chapter 12 automatic checks passed",
    r"\mathbb E[\mathbb E[X\mid\mathcal G]]=\mathbb E[X]",
    r"\mathbb E[ZX]=\mathbb E[Z\mathbb E[X\mid\mathcal G]]",
    r"\operatorname{Var}(X)=\mathbb E[\operatorname{Var}(X\mid\mathcal G)]+\operatorname{Var}(\mathbb E[X\mid\mathcal G])",
    r"\mathbb E[X\mid Y]=m(Y)",
    r"\operatorname{Var}(X\mid Y)=\sigma_X^2(1-\rho^2)\quad\text{for a bivariate normal pair}",
    note=(
        "Partition conditioning, total expectation, Jensen, MMSE, tower-counterexample, "
        "conditional-density and bivariate-normal checks all passed."
    ),
)


## 30. Chapter map

| Chapter concept | Computational representation |
|---|---|
| information-dependent mean | even/odd die cells |
| event conditioning | exact quotient formula |
| partition conditioning | cellwise averages |
| abstract definition | testing identity with $\mathbf 1_A$ |
| expectation/integral notation | equivalent formulations |
| existence/uniqueness | Radon--Nikodym and versions |
| conditional probability | conditional indicator expectation |
| basic properties | mean preservation, known quantities, contraction |
| conditional convergence | finite monotone approximation illustration |
| testing identity | known indicator multiplier |
| taking out what is known | information-measurable factors |
| tower property | nested information |
| total expectation | die and three-risk-class examples |
| filtration | two-stage coin information |
| independent information | unchanged conditional mean |
| false converse | $Y=X^2$ |
| conditional Jensen | parity-partition square check |
| conditional variance | two-dice decomposition |
| total covariance | shared-signal model |
| MMSE prediction | exact two-dice MSE comparison |
| conditioning on $Y$ | $\sigma(Y)$ and Doob--Dynkin |
| countable $Y$ | two-dice grouped conditional means |
| conditional density | triangular model |
| density-form total expectation | $E[X]=1/3$ in triangle |
| second density example | $Y\mid X=x\sim U(0,x)$ |
| bivariate normal | affine conditional mean and constant conditional variance |
| Galton | regression toward the mean |
| model risk | correct conditioning does not validate the model |
| AI Audit | explicit non-nested tower counterexample |

The chapter's central viewpoint is:

$$
\boxed{
\text{conditional expectation}
=
\text{best average compatible with retained information}.
}
$$

Its density formulas are concrete representations of this abstract object, not separate definitions.
